# Smart Inventory Analytics & Reorder Point Optimization
> **Domain**: Enterprise Systems / Inventory Analytics | **Tech Stack**: Python, SQLite, Pandas, NumPy
> **Author**: [Arjuna Fransesco](https://github.com/ArjunaFransesco) | **GitHub**: [Portfolio Repositories](https://github.com/ArjunaFransesco?tab=repositories)

---
## Executive Summary
Data engineering and analytics notebook exploring warehouse stock valuation, gross margin distributions, and Economic Order Quantity (EOQ) safety stock modeling.

---
## Machine Learning Workflow
1. **Domain Understanding & Dataset Ingestion**
2. **Exploratory Data Analysis (EDA) & Feature Distribution**
3. **Preprocessing Pipeline & Feature Engineering**
4. **Model Architecture & Training**
5. **Evaluation, Error Metrics & Performance Visualization**
6. **Deployment Artifact Export**


### 1. Ingesting Warehouse Inventory Data & Computing Key Metrics


In [1]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect('../data/inventory_pos.db')
df_products = pd.read_sql_query("""
    SELECT p.sku, p.name, c.name as category, p.cost_price, p.selling_price, 
           p.stock_quantity, p.min_reorder_level
    FROM products p
    JOIN categories c ON p.category_id = c.id
""", conn)

df_products['stock_value'] = df_products['cost_price'] * df_products['stock_quantity']
df_products['gross_margin_pct'] = ((df_products['selling_price'] - df_products['cost_price']) / df_products['selling_price']) * 100
df_products['reorder_status'] = np.where(df_products['stock_quantity'] <= df_products['min_reorder_level'], 'REORDER REQUIRED', 'HEALTHY')

print(f"Total Catalog Stock Value: Rp {df_products['stock_value'].sum():,.0f}")
print(df_products[['sku', 'name', 'stock_quantity', 'min_reorder_level', 'gross_margin_pct', 'reorder_status']].head(10))



### 2. Economic Order Quantity (EOQ) & Safety Stock Optimization Formulation
$$\text{EOQ} = \sqrt{\frac{2DS}{H}}$$
Where $D$ is annual demand, $S$ is order cost, and $H$ is holding cost per unit per year.


In [1]:
# Economic Order Quantity calculation benchmark
annual_demand = 1200  # units
order_cost = 50000    # Rp per order
holding_cost = 12000  # Rp per unit/year

eoq = np.sqrt((2 * annual_demand * order_cost) / holding_cost)
print(f"Optimal Economic Order Quantity (EOQ): {eoq:.1f} units per batch")

